# Lesson 16: Context Engineering & Root/Sampling Migration

In this notebook, you will:
1. Implement a token-budgeted **Context Assembler** for MCP resources.
2. Enforce `Path.resolve().is_relative_to()` workspace sandboxing.
3. Compute Context Utilization Ratio (CUR) over a `gemini-2.5-flash` response.

In [ ]:
!pip install -q -U google-genai

In [ ]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir).resolve()
    safe_file = (root / "notes.md").resolve()
    safe_file.write_text("MCP Context Engineering notes")

    def safe_read(rel_path: str) -> str:
        target = (root / rel_path).resolve()
        if not target.is_relative_to(root):
            raise PermissionError(f"Blocked traversal outside workspace: {rel_path}")
        return target.read_text()

    print("Safe read:", safe_read("notes.md"))
    try:
        safe_read("../../etc/passwd")
    except PermissionError as exc:
        print("Security guard:", exc)

### Challenge Exercise
Combine the safe file reader with a progressive context packer that summarizes older conversation turns once total context exceeds `2,000` characters.